In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA student_gold;

CREATE OR REPLACE VIEW workspace.student_gold.vw_student_performance AS

SELECT
    f.*,

    -- Student attributes
    s.student_id,
    s.age,
    s.age_group,
    s.gender,
    s.education_level,
    s.school_type,
    s.family_income,
    s.parent_education,
    s.urban_rural,

    -- Study profile
    st.class_participation,
    st.private_tuition,
    st.study_consistency,
    st.study_environment,
    st.study_method,
    st.revision_frequency,
    st.notes_quality,

    -- Lifestyle profile
    l.sleep_quality,
    l.break_frequency,
    l.motivation_level,

    -- Digital access
    d.internet_access,
    d.device_availability,
    d.educational_app_usage

FROM workspace.student_gold.fact_exam_performance f

INNER JOIN workspace.student_gold.dim_student s
    ON f.student_key = s.student_key

INNER JOIN workspace.student_gold.dim_study_profile st
    ON f.study_profile_key = st.study_profile_key

INNER JOIN workspace.student_gold.dim_lifestyle_profile l
    ON f.lifestyle_key = l.lifestyle_key

INNER JOIN workspace.student_gold.dim_digital_access d
    ON f.digital_key = d.digital_key;

In [0]:
%sql
-- Academic Performance KPI

CREATE OR REPLACE VIEW workspace.student_gold.agg_academic_performance AS

SELECT
    COUNT(*) AS student_count,

    COUNT_IF(is_passed) AS passed_students,

    COUNT_IF(NOT is_passed) AS failed_students,

    ROUND(AVG(exam_score), 2)
        AS avg_exam_score,

    ROUND(MEDIAN(exam_score), 2)
        AS median_exam_score,

    ROUND(AVG(previous_exam_score), 2)
        AS avg_previous_exam_score,

    ROUND(AVG(exam_score_change), 2)
        AS avg_score_change,

    ROUND(AVG(CAST(is_passed AS INT)) * 100, 2)
        AS pass_rate_pct,

    ROUND(AVG(study_hours_per_day), 2)
        AS avg_study_hours,

    ROUND(AVG(attendance_percentage), 2)
        AS avg_attendance,

    ROUND(AVG(sleep_hours), 2)
        AS avg_sleep_hours

FROM workspace.student_gold.fact_exam_performance;

In [0]:
%sql

-- study habbits

CREATE OR REPLACE VIEW workspace.student_gold.agg_study_habits AS

SELECT
    study_hours_group,
    study_method,

    COUNT(*) AS student_count,

    ROUND(AVG(exam_score), 2)
        AS avg_exam_score,

    ROUND(AVG(study_hours_per_day), 2)
        AS avg_study_hours,

    ROUND(AVG(self_study_hours), 2)
        AS avg_self_study_hours,

    ROUND(AVG(exam_score_change), 2)
        AS avg_score_change,

    ROUND(AVG(CAST(is_passed AS INT)) * 100, 2)
        AS pass_rate_pct

FROM workspace.student_gold.vw_student_performance

GROUP BY
    study_hours_group,
    study_method;

In [0]:
%sql

-- lifestyle impact

CREATE OR REPLACE VIEW workspace.student_gold.agg_lifestyle_impact AS

SELECT
    sleep_hours_group,
    stress_level,

    COUNT(*) AS student_count,

    ROUND(AVG(exam_score), 2)
        AS avg_exam_score,

    ROUND(AVG(sleep_hours), 2)
        AS avg_sleep_hours,

    ROUND(AVG(daily_screen_time), 2)
        AS avg_screen_time,

    ROUND(AVG(physical_activity_hours), 2)
        AS avg_physical_activity,

    ROUND(AVG(exam_anxiety_level), 2)
        AS avg_exam_anxiety,

    ROUND(AVG(CAST(is_passed AS INT)) * 100, 2)
        AS pass_rate_pct

FROM workspace.student_gold.fact_exam_performance

GROUP BY
    sleep_hours_group,
    stress_level;

In [0]:
%sql

--Socioeconomic Factors

CREATE OR REPLACE VIEW workspace.student_gold.agg_socioeconomic_factors AS

SELECT
    family_income,
    parent_education,
    school_type,

    COUNT(*) AS student_count,

    ROUND(AVG(exam_score), 2)
        AS avg_exam_score,

    ROUND(AVG(previous_gpa), 2)
        AS avg_previous_gpa,

    ROUND(AVG(attendance_percentage), 2)
        AS avg_attendance,

    ROUND(AVG(study_hours_per_day), 2)
        AS avg_study_hours,

    ROUND(AVG(CAST(private_tuition AS INT)) * 100, 2)
        AS private_tuition_pct,

    ROUND(AVG(CAST(is_passed AS INT)) * 100, 2)
        AS pass_rate_pct

FROM workspace.student_gold.vw_student_performance

GROUP BY
    family_income,
    parent_education,
    school_type;

In [0]:
%sql

-- digital learning

CREATE OR REPLACE VIEW workspace.student_gold.agg_digital_learning AS

SELECT
    internet_access,
    device_availability,
    educational_app_usage,

    COUNT(*) AS student_count,

    ROUND(AVG(exam_score), 2)
        AS avg_exam_score,

    ROUND(AVG(online_learning_hours), 2)
        AS avg_online_learning_hours,

    ROUND(AVG(online_course_hours), 2)
        AS avg_online_course_hours,

    ROUND(AVG(daily_screen_time), 2)
        AS avg_screen_time,

    ROUND(AVG(CAST(is_passed AS INT)) * 100, 2)
        AS pass_rate_pct

FROM workspace.student_gold.vw_student_performance

GROUP BY
    internet_access,
    device_availability,
    educational_app_usage;

In [0]:
%sql

-- exam_preparation

CREATE OR REPLACE VIEW workspace.student_gold.agg_exam_preparation AS

WITH preparation AS (

    SELECT
        *,

        CASE
            WHEN exam_preparation_days IS NULL THEN 'Unknown'
            WHEN exam_preparation_days <= 7 THEN '1-7 days'
            WHEN exam_preparation_days <= 14 THEN '8-14 days'
            WHEN exam_preparation_days <= 21 THEN '15-21 days'
            ELSE '22+ days'
        END AS preparation_group,

        CASE
            WHEN practice_tests_completed IS NULL THEN 'Unknown'
            WHEN practice_tests_completed = 0 THEN '0 tests'
            WHEN practice_tests_completed <= 5 THEN '1-5 tests'
            WHEN practice_tests_completed <= 10 THEN '6-10 tests'
            ELSE '11+ tests'
        END AS practice_tests_group

    FROM workspace.student_gold.fact_exam_performance
)

SELECT
    preparation_group,
    practice_tests_group,
    exam_difficulty,

    COUNT(*) AS student_count,

    ROUND(AVG(exam_score), 2)
        AS avg_exam_score,

    ROUND(AVG(question_accuracy_pct), 2)
        AS avg_question_accuracy,

    ROUND(AVG(time_management_score), 2)
        AS avg_time_management,

    ROUND(AVG(exam_anxiety_level), 2)
        AS avg_exam_anxiety,

    ROUND(AVG(CAST(is_passed AS INT)) * 100, 2)
        AS pass_rate_pct

FROM preparation

GROUP BY
    preparation_group,
    practice_tests_group,
    exam_difficulty;

In [0]:
%sql

-- grade distribution

CREATE OR REPLACE VIEW workspace.student_gold.agg_grade_distribution AS

SELECT
    performance_grade,

    COUNT(*) AS student_count,

    ROUND(
        100.0 * COUNT(*) /
        SUM(COUNT(*)) OVER (),
        2
    ) AS student_percentage,

    ROUND(AVG(exam_score), 2)
        AS avg_exam_score

FROM workspace.student_gold.fact_exam_performance

GROUP BY performance_grade;

In [0]:
%sql
SHOW TABLES IN workspace.student_gold;
SELECT *
FROM workspace.student_gold.agg_academic_performance;

student_count,passed_students,failed_students,avg_exam_score,median_exam_score,avg_previous_exam_score,avg_score_change,pass_rate_pct,avg_study_hours,avg_attendance,avg_sleep_hours
100000,77389,22611,61.95,61.96,69.53,-7.58,77.39,3.0,84.62,7.0


In [0]:
%sql

-- data validation

SELECT
    'fact_exam_performance' AS object_name,
    COUNT(*) AS total_students
FROM workspace.student_gold.fact_exam_performance

UNION ALL

SELECT
    'vw_student_performance',
    COUNT(*)
FROM workspace.student_gold.vw_student_performance

UNION ALL

SELECT
    'agg_academic_performance',
    SUM(student_count)
FROM workspace.student_gold.agg_academic_performance

UNION ALL

SELECT
    'agg_study_habits',
    SUM(student_count)
FROM workspace.student_gold.agg_study_habits

UNION ALL

SELECT
    'agg_lifestyle_impact',
    SUM(student_count)
FROM workspace.student_gold.agg_lifestyle_impact

UNION ALL

SELECT
    'agg_socioeconomic_factors',
    SUM(student_count)
FROM workspace.student_gold.agg_socioeconomic_factors

UNION ALL

SELECT
    'agg_digital_learning',
    SUM(student_count)
FROM workspace.student_gold.agg_digital_learning

UNION ALL

SELECT
    'agg_exam_preparation',
    SUM(student_count)
FROM workspace.student_gold.agg_exam_preparation

UNION ALL

SELECT
    'agg_grade_distribution',
    SUM(student_count)
FROM workspace.student_gold.agg_grade_distribution;

object_name,total_students
vw_student_performance,100000
agg_academic_performance,100000
agg_study_habits,100000
agg_lifestyle_impact,100000
agg_socioeconomic_factors,100000
agg_digital_learning,100000
agg_exam_preparation,100000
agg_grade_distribution,100000
fact_exam_performance,100000


In [0]:
%sql
SELECT *
FROM workspace.student_gold.agg_study_habits
ORDER BY avg_exam_score DESC;

study_hours_group,study_method,student_count,avg_exam_score,avg_study_hours,avg_self_study_hours,avg_score_change,pass_rate_pct
6h+,Self-Reading,1197,76.74,7.39,5.14,7.24,95.99
6h+,Practice Tests,1242,76.1,7.3,5.14,6.11,95.49
6h+,Group Study,1221,75.72,7.38,5.19,6.57,95.09
6h+,Summarizing,1221,75.27,7.27,5.09,5.75,95.33
6h+,Flashcards,1214,75.21,7.25,5.07,5.37,95.3
4-6h,Self-Reading,3516,68.29,4.81,3.35,-1.31,90.16
4-6h,Flashcards,3497,68.19,4.81,3.37,-1.36,88.45
4-6h,Group Study,3575,68.17,4.8,3.35,-1.18,88.39
4-6h,Practice Tests,3407,68.16,4.81,3.36,-1.26,88.7
4-6h,Summarizing,3581,67.94,4.8,3.34,-1.44,88.47
